In [21]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [22]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas_ta as ta
import pygame
import pytz
import json

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque

In [23]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 75  # 75 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 30  # 30 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [24]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [25]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [26]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [27]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [28]:
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame):
        # df must have columns [datetime, open, high, low, close, (volume optional)]
        self.df = df.copy()

    def preprocess_datetime(self):
        # Convert Unix timestamp to local time (Asia/Kolkata)
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates or missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set as index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)

        return self

    def clean_data(self):
        # Drop volume if zero or NaN
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        # RSI
        self.df.sort_index(inplace=True)
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR
        self.df['atr_base'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )

        return self

    def add_swing_points(self, left_bars=10, right_bars=10):
        # Identify local swing highs/lows
        self.df.sort_index(inplace=True)

        rolling_high = self.df['high'].rolling(left_bars + 1, center=False).max()
        rolling_low = self.df['low'].rolling(left_bars + 1, center=False).min()

        forward_high = self.df['high'].shift(-right_bars).rolling(right_bars + 1, center=False).max()
        forward_low = self.df['low'].shift(-right_bars).rolling(right_bars + 1, center=False).min()

        self.df['is_swing_high'] = 0.0
        cond_high = (self.df['high'] == rolling_high) & (self.df['high'] == forward_high)
        self.df.loc[cond_high, 'is_swing_high'] = 1.0

        self.df['is_swing_low'] = 0.0
        cond_low = (self.df['low'] == rolling_low) & (self.df['low'] == forward_low)
        self.df.loc[cond_low, 'is_swing_low'] = 1.0

        return self

    def add_range_breakout_features(self, window=20):
        # Donchian Channels
        self.df.sort_index(inplace=True)

        self.df['donchian_high'] = self.df['high'].rolling(window).max()
        self.df['donchian_low'] = self.df['low'].rolling(window).min()
        self.df['donchian_range'] = self.df['donchian_high'] - self.df['donchian_low']

        # Donchian breakout flags
        self.df['donchian_breakout'] = np.where(
            self.df['close'] > self.df['donchian_high'], 1.0,
            np.where(self.df['close'] < self.df['donchian_low'], -1.0, 0.0)
        )

        # Range expansion measure
        self.df['range_expansion'] = (self.df['high'] - self.df['low']).rolling(window).std()

        return self

    def add_momentum_features(self):
        self.df.sort_index(inplace=True)

        # Stoch
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14, d=3
        )
        self.df['stoch_k'] = stoch['STOCHk_14_3_3']
        self.df['stoch_d'] = stoch['STOCHd_14_3_3']

        # ADX
        adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
        self.df['adx'] = adx_data['ADX_14']
        self.df['diplus'] = adx_data['DMP_14']
        self.df['diminus'] = adx_data['DMN_14']

        # CCI
        self.df['cci'] = ta.cci(self.df['high'], self.df['low'], self.df['close'], length=20)

        return self

    def add_volatility_features(self, window=20):
        # Historical volatility, Z-score
        self.df.sort_index(inplace=True)

        self.df['log_return'] = np.log(self.df['close'] / self.df['close'].shift(1))
        self.df['hist_vol'] = self.df['log_return'].rolling(window).std() * np.sqrt(252)

        rolling_mean = self.df['log_return'].rolling(window).mean()
        rolling_std = self.df['log_return'].rolling(window).std()
        self.df['zscore_return'] = (self.df['log_return'] - rolling_mean) / (rolling_std + 1e-9)

        return self

    def add_volume_features(self):
        # Volume-based features if present
        if 'volume' not in self.df.columns:
            return self

        self.df.sort_index(inplace=True)

        # OBV
        self.df['obv'] = ta.obv(self.df['close'], self.df['volume'])

        # Volume spikes
        vol_mean = self.df['volume'].rolling(20).mean()
        vol_std = self.df['volume'].rolling(20).std()
        self.df['vol_spike'] = np.where(
            (self.df['volume'] > (vol_mean + 2 * vol_std)), 1.0, 0.0
        )

        # VWAP approximation
        typical_price = (self.df['high'] + self.df['low'] + self.df['close']) / 3.0
        self.df['cum_tp_vol'] = (typical_price * self.df['volume']).cumsum()
        self.df['cum_vol'] = self.df['volume'].cumsum()
        self.df['vwap'] = self.df['cum_tp_vol'] / (self.df['cum_vol'] + 1e-9)

        self.df.drop(['cum_tp_vol', 'cum_vol'], axis=1, inplace=True)

        return self

    def add_regime_features(self):
        # Classify trending vs ranging via ADX threshold
        self.df.sort_index(inplace=True)

        if 'adx' not in self.df.columns:
            adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
            self.df['adx'] = adx_data['ADX_14']

        self.df['regime_trend'] = np.where(self.df['adx'] >= 25, 1.0, 0.0)

        return self

    def add_time_features(self):
        # Hour, day-of-week, cyclical encodings
        self.df.sort_index(inplace=True)

        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)

        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)

        return self

    def add_adaptive_targets_and_stops(self):
        # Adapt targets/stops based on regime
        self.df.sort_index(inplace=True)

        if 'atr_base' not in self.df.columns:
            self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        if 'regime_trend' not in self.df.columns:
            self.add_regime_features()

        is_trend = (self.df['regime_trend'] == 1.0)
        self.df['Target'] = np.where(is_trend, 3.0 * self.df['atr_base'], 1.5 * self.df['atr_base'])
        self.df['StopLoss'] = np.where(is_trend, 1.5 * self.df['atr_base'], 1.0 * self.df['atr_base'])

        return self

    def label_signals(self):
        """
        Priority is profit first (buy_target/sell_target),
        then stop-loss (buy_sl/sell_sl).
        """
        self.df['Signal'] = 0
        self.df['Entry Price'] = 0.0
        self.df['Exit Price'] = 0.0
        self.df['candles_to_profit'] = 0
        self.df['candles_to_loss'] = 0

        all_idx = self.df.index.tolist()
        n = len(all_idx)

        for i in range(n - 1):
            idx_i = all_idx[i]

            entry_price = self.df.at[idx_i, 'close']
            target = self.df.at[idx_i, 'Target']
            stop_loss = self.df.at[idx_i, 'StopLoss']

            # Hypothetical buy/sell levels
            buy_target_price = entry_price + target
            buy_sl_price = entry_price - stop_loss
            sell_target_price = entry_price - target
            sell_sl_price = entry_price + stop_loss

            future_slice = all_idx[i+1:]
            signal_found = False

            for offset, future_idx in enumerate(future_slice, start=1):
                future_high = self.df.at[future_idx, 'high']
                future_low = self.df.at[future_idx, 'low']

                triggers = []
                # Check buy/sell target/SL
                if future_high >= buy_target_price:
                    triggers.append(('buy_target', offset))
                if future_low <= buy_sl_price:
                    triggers.append(('buy_sl', offset))
                if future_low <= sell_target_price:
                    triggers.append(('sell_target', offset))
                if future_high >= sell_sl_price:
                    triggers.append(('sell_sl', offset))

                # If multiple triggers this candle, pick the earliest by priority
                if triggers:
                    priority_map = {
                        'buy_target': 1,
                        'sell_target': 2,
                        'buy_sl': 3,
                        'sell_sl': 4
                    }
                    triggers.sort(key=lambda x: priority_map[x[0]])
                    first_trigger, candle_count = triggers[0]

                    if first_trigger == 'buy_target':
                        self.df.at[idx_i, 'Signal'] = 1
                        self.df.at[idx_i, 'Exit Price'] = buy_target_price
                        self.df.at[idx_i, 'candles_to_profit'] = candle_count

                    elif first_trigger == 'sell_target':
                        self.df.at[idx_i, 'Signal'] = 3
                        self.df.at[idx_i, 'Exit Price'] = sell_target_price
                        self.df.at[idx_i, 'candles_to_profit'] = candle_count

                    elif first_trigger == 'buy_sl':
                        self.df.at[idx_i, 'Signal'] = 2
                        self.df.at[idx_i, 'Exit Price'] = buy_sl_price
                        self.df.at[idx_i, 'candles_to_loss'] = candle_count

                    elif first_trigger == 'sell_sl':
                        self.df.at[idx_i, 'Signal'] = 4
                        self.df.at[idx_i, 'Exit Price'] = sell_sl_price
                        self.df.at[idx_i, 'candles_to_loss'] = candle_count

                    self.df.at[idx_i, 'Entry Price'] = entry_price
                    signal_found = True
                    break

            # If no trigger in future
            if not signal_found:
                self.df.at[idx_i, 'Signal'] = 0
                self.df.at[idx_i, 'Entry Price'] = entry_price

        return self

    def run_pipeline(self):
        (
            self.preprocess_datetime()
                .clean_data()
                .add_base_timeframe_features()
                .add_swing_points()
                .add_range_breakout_features()
                .add_momentum_features()
                .add_volatility_features()
                .add_volume_features()
                .add_regime_features()
                .add_time_features()
                .add_adaptive_targets_and_stops()
                .label_signals()
        )
        return self

    def get_processed_df(self):
        # Drop rows that have NaN from lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        return self.df

In [29]:
nf_index_symbol, nf_quantity = get_index_symbol_and_quantity("Nifty")
bnf_index_symbol, bnf_quantity = get_index_symbol_and_quantity("Bank Nifty")

#nf_train_df = fetch_candle_data(100, nf_index_symbol, two_interval_minutes)
#bnf_train_df = fetch_candle_data(100, bnf_index_symbol, two_interval_minutes)

# Fetch candle data for each instrument and each timeframe
nifty_2m = fetch_train_candle_data(3, nf_index_symbol, 2)
nifty_5m = fetch_train_candle_data(3, nf_index_symbol, 5)
nifty_15m = fetch_train_candle_data(3, nf_index_symbol, 15)

bnf_2m = fetch_train_candle_data(3, bnf_index_symbol, 2)
bnf_5m = fetch_train_candle_data(3, bnf_index_symbol, 5)
bnf_15m = fetch_train_candle_data(3, bnf_index_symbol, 15)

print("Fetched historical data for all instruments")

# Clean and remove duplicate datetimes (if any)
nifty_2m = nifty_2m.drop_duplicates(subset='datetime', keep='first')
nifty_5m = nifty_5m.drop_duplicates(subset='datetime', keep='first')
nifty_15m = nifty_15m.drop_duplicates(subset='datetime', keep='first')

bnf_2m = bnf_2m.drop_duplicates(subset='datetime', keep='first')
bnf_5m = bnf_5m.drop_duplicates(subset='datetime', keep='first')
bnf_15m = bnf_15m.drop_duplicates(subset='datetime', keep='first')

# Nifty after processing
nf_train_pipeline_2m = FullFeaturePipeline(nifty_2m)
nf_train_pipeline_2m.run_pipeline()
df_nifty_2m = nf_train_pipeline_2m.get_processed_df()

nf_train_pipeline_5m = FullFeaturePipeline(nifty_5m)
nf_train_pipeline_5m.run_pipeline()
df_nifty_5m = nf_train_pipeline_5m.get_processed_df()

nf_train_pipeline_15m = FullFeaturePipeline(nifty_15m)
nf_train_pipeline_15m.run_pipeline()
df_nifty_15m = nf_train_pipeline_15m.get_processed_df()

# Bank Nifty after processing
bnf_train_pipeline_2m = FullFeaturePipeline(bnf_2m)
bnf_train_pipeline_2m.run_pipeline()
df_bnf_2m = bnf_train_pipeline_2m.get_processed_df()

bnf_train_pipeline_5m = FullFeaturePipeline(bnf_5m)
bnf_train_pipeline_5m.run_pipeline()
df_bnf_5m = bnf_train_pipeline_5m.get_processed_df()

bnf_train_pipeline_15m = FullFeaturePipeline(bnf_15m)
bnf_train_pipeline_15m.run_pipeline()
df_bnf_15m = bnf_train_pipeline_15m.get_processed_df()

Fetched historical data for all instruments


In [30]:
df_bnf_2m.columns

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Index(['open', 'high', 'low', 'close', 'rsi_base', 'macd', 'macd_h', 'macd_s',
       'bb_lower', 'bb_mid', 'bb_upper', 'atr_base', 'is_swing_high',
       'is_swing_low', 'donchian_high', 'donchian_low', 'donchian_range',
       'donchian_breakout', 'range_expansion', 'stoch_k', 'stoch_d', 'adx',
       'diplus', 'diminus', 'cci', 'log_return', 'hist_vol', 'zscore_return',
       'regime_trend', 'hour', 'day_of_week', 'hour_sin', 'hour_cos',
       'dow_sin', 'dow_cos', 'Target', 'StopLoss', 'Signal',
       'candles_to_profit', 'candles_to_loss'],
      dtype='object')

In [31]:
df_bnf_2m

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss,Signal,candles_to_profit,candles_to_loss
datetime,,,,,,,,,,,,,,,,,,,,,
2023-12-29 10:21:00,48249.20,48275.90,48248.40,48264.50,44.517764,-20.154046,-3.635998,-16.518048,48209.115989,48287.0500,...,4.0,0.500000,-0.866025,-0.433884,-0.900969,51.079157,34.052771,2,0,3
2023-12-29 10:23:00,48265.40,48291.20,48252.10,48277.80,47.917897,-17.685913,-0.934292,-16.751621,48208.433533,48286.4600,...,4.0,0.500000,-0.866025,-0.433884,-0.900969,51.667266,34.444844,2,0,1
2023-12-29 10:25:00,48275.60,48281.20,48239.80,48239.80,40.315794,-18.581981,-1.464288,-17.117693,48203.478445,48284.1100,...,4.0,0.500000,-0.866025,-0.433884,-0.900969,52.472654,34.981769,2,0,2
2023-12-29 10:27:00,48240.10,48264.10,48220.10,48226.80,38.089484,-20.109304,-2.393289,-17.716015,48196.685360,48279.7150,...,4.0,0.500000,-0.866025,-0.433884,-0.900969,53.510949,35.673966,2,0,13
2023-12-29 10:29:00,48220.70,48234.10,48195.90,48210.70,35.476604,-22.361090,-3.716060,-18.645030,48187.891995,48274.5850,...,4.0,0.500000,-0.866025,-0.433884,-0.900969,53.800238,35.866825,4,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-24 15:21:00,51557.45,51569.45,51555.50,51560.45,58.278447,9.039931,11.331906,-2.291975,51408.128097,51500.3525,...,3.0,-0.707107,-0.707107,0.433884,-0.900969,49.293984,32.862656,2,0,3
2024-10-24 15:23:00,51559.00,51571.25,51542.30,51561.50,58.497006,10.375452,10.133942,0.241511,51408.411057,51504.1075,...,3.0,-0.707107,-0.707107,0.433884,-0.900969,48.874771,32.583181,2,0,2
2024-10-24 15:25:00,51559.80,51566.80,51547.65,51554.55,56.391283,10.749146,8.406108,2.343038,51413.453868,51508.9875,...,3.0,-0.707107,-0.707107,0.433884,-0.900969,47.435502,31.623668,2,0,1


In [48]:
class Position:
    def __init__(self):
        self.reset()

    def reset(self):
        self.status = "flat"  # "flat", "long", or "short"
        self.entry_price = None
        self.sl_points = None
        self.target_points = None
        self.direction = 0  # +1 for long, -1 for short
        self.quantity = 0
        self.time_in_trade = 0

    def update_unrealized_pnl(self, current_price: float) -> float:
        return (current_price - self.entry_price) * self.direction * self.quantity

    def open(self, entry_price: float, sl_points: float, target_points: float,
             direction: int, quantity: int):
        self.status = "long" if direction == 1 else "short"
        self.entry_price = entry_price
        self.sl_points = sl_points
        self.target_points = target_points
        self.direction = direction
        self.quantity = quantity
        self.time_in_trade = 0

class TradingEnv(gym.Env):
    def __init__(self, env_config):
        super(TradingEnv, self).__init__()
        self.window_size = env_config["window_size"]
        # Retain guidance columns for training; drop them from the agent's observation.
        self.df_guidance = env_config["df"][['Signal', 'candles_to_profit', 'candles_to_loss']].copy()
        self.df = env_config["df"].drop(columns=['Signal', 'candles_to_profit', 'candles_to_loss']).copy()
        self.lot_size = env_config["lot_size"]  # e.g. 75 for NIFTY, 1 for Bitcoin.
        self.transaction_cost = env_config["transaction_cost"]  # e.g. 20.0 units per trade.
        self.instrument_name = env_config.get("name", "Unknown")

        # Normalize initial capital across instruments.
        self.initial_capital = self.df['high'].max() * self.lot_size * 3
        self.current_capital = None
        self.current_step = None
        self.position = Position()
        self.trade_log = []

        # Action space: 0 = Hold, 1 = Buy, 2 = Sell, 3 = Close.
        self.action_space = spaces.Discrete(4)

        # Observation space: The agent sees:
        # - "market": current market features (without guidance columns)
        # - "position": a 4-dimensional vector (one-hot long, one-hot short, normalized unrealized PnL, time held)
        # - "history": a window of past market features.
        n_features = len(self.df.columns)
        self.observation_space = spaces.Dict({
            "market": spaces.Box(-np.inf, np.inf, shape=(n_features,), dtype=np.float32),
            "position": spaces.Box(-np.inf, np.inf, shape=(4,), dtype=np.float32),
            "history": spaces.Box(-np.inf, np.inf, shape=(self.window_size, n_features), dtype=np.float32)
        })

        # Reward shaping parameters:
        self.unrealized_pnl_weight = env_config.get("unrealized_pnl_weight", 0.1)  # e.g. 0.1
        self.time_penalty_weight = env_config.get("time_penalty_weight", 0.02)       # e.g. 0.02 per 100 steps
        self.volatility_penalty_weight = env_config.get("volatility_penalty_weight", 0.1)  # e.g. 0.1

        # Guidance reward parameters (for training only)
        self.target_bonus = env_config.get("target_bonus", 0.5)                    # e.g. 0.5 bonus
        self.sl_penalty = env_config.get("sl_penalty", 0.5)                        # e.g. 0.5 penalty
        self.misalignment_penalty = env_config.get("misalignment_penalty", 0.5)      # (scaled below)
        self.missed_opportunity_penalty = env_config.get("missed_opportunity_penalty", 0.05)  # e.g. 0.05

    def reset(self, seed=None, options=None):
        self.current_step = self.window_size
        self.position.reset()
        self.current_capital = self.initial_capital
        self.trade_log = []
        return self._get_observation(), {}

    def step(self, action: int):
        current_data = self.df.iloc[self.current_step]
        guidance_row = self.df_guidance.iloc[self.current_step]
        reward = 0.0

        #########################################
        # Immediate Guidance Reward (Only When Flat)
        #########################################
        if self.position.status == "flat":
            signal = guidance_row['Signal']
            candles_to_profit = guidance_row['candles_to_profit']
            candles_to_loss = guidance_row['candles_to_loss']

            # Determine correct action: For signals 1,2 (buy outcomes) => Buy (action 1); for 3,4 (sell outcomes) => Sell (action 2)
            if signal in [1, 2]:
                correct_action = 1
            elif signal in [3, 4]:
                correct_action = 2
            else:
                correct_action = None  # Should not happen

            # If action is aligned, reward or modest penalty is scaled by the candle count.
            if action == correct_action:
                if signal in [1, 3]:  # target hit expected
                    guidance_reward = self.target_bonus / candles_to_profit
                elif signal in [2, 4]:  # stop-loss expected
                    guidance_reward = -self.sl_penalty / candles_to_loss
            else:
                # Scale misalignment penalty using the same candle information.
                if signal in [1, 3]:
                    guidance_reward = -self.target_bonus / candles_to_profit
                elif signal in [2, 4]:
                    guidance_reward = -self.sl_penalty / candles_to_loss
            reward += guidance_reward

        #########################################
        # Check for Stop-Loss Hit (Position Open)
        #########################################
        if self._check_sl_hit(current_data):
            reward += self._close_position(current_data, "SL Hit")

        #########################################
        # Execute Agent Action (Open, Reverse, or Close Position)
        #########################################
        self._execute_action(action, current_data)

        # Increment time if a position is open.
        if self.position.status != "flat":
            self.position.time_in_trade += 1

        # Continuous shaping reward when position is open.
        if self.position.status != "flat":
            unrealized_pnl = self.position.update_unrealized_pnl(current_data['close'])
            reward += self._calculate_reward(unrealized_pnl, current_data)

        # Advance simulation.
        self.current_step += 1
        if self.current_step >= len(self.df) or self.current_capital <= 0:
            done = True
            if self.position.status != "flat":
                reward += self._close_position(current_data, "Force Close")
        else:
            done = False

        return self._get_observation(), reward, done, False, {}

    def _get_observation(self) -> dict:
        # Ensure the index is within bounds
        idx = min(self.current_step, len(self.df) - 1)

        # Market features (without guidance columns)
        market_features = self.df.iloc[idx].values.astype(np.float32)

        # Position context: one-hot for long/short, normalized unrealized PnL, normalized time held.
        position_context = np.array([
            1.0 if self.position.status == "long" else 0.0,
            1.0 if self.position.status == "short" else 0.0,
            (self.position.update_unrealized_pnl(self.df.iloc[idx]['close'])
            if self.position.status != "flat" else 0.0) / self.initial_capital,
            self.position.time_in_trade / 100.0
        ], dtype=np.float32)

        # Historical market data window.
        history_start = max(0, idx - self.window_size)
        history = self.df.iloc[history_start:idx].values.astype(np.float32)
        if len(history) < self.window_size:
            padding = np.zeros((self.window_size - len(history), self.df.shape[1]), dtype=np.float32)
            history = np.vstack([padding, history])
        return {"market": market_features, "position": position_context, "history": history}

    def _execute_action(self, action: int, current_data: pd.Series):
        price = current_data['open']
        # Action 3 (Close) only valid if a position exists.
        if action == 3 and self.position.status != "flat":
            self._close_position(current_data, "Manual Close")
        elif action == 1:
            # Buy action. If currently short, reverse.
            if self.position.status == "short":
                self._close_position(current_data, "Reverse Close")
            if self.position.status == "flat" and self.current_capital >= price * self.lot_size:
                self._open_position("long", current_data)
        elif action == 2:
            # Sell action. If currently long, reverse.
            if self.position.status == "long":
                self._close_position(current_data, "Reverse Close")
            if self.position.status == "flat" and self.current_capital >= price * self.lot_size:
                self._open_position("short", current_data)
        # If action is 0 (Hold), do nothing.

    def _open_position(self, direction: str, current_data: pd.Series):
        entry_price = current_data['open']
        # Deduct cost immediately.
        self.current_capital -= entry_price * self.lot_size
        self.position.open(
            entry_price=entry_price,
            sl_points=current_data['StopLoss'],
            target_points=current_data['Target'],
            direction=1 if direction == "long" else -1,
            quantity=self.lot_size
        )
        self.trade_log.append({
            "entry_time": current_data.name,
            "position": direction,
            "entry_price": entry_price,
            "exit_time": None,
            "exit_price": None,
            "pnl": None,
            "reason": None
        })

    def _close_position(self, current_data: pd.Series, reason: str) -> float:
        if self.position.status == "flat":
            return 0.0
        # If stop-loss hit, use SL price; otherwise, use current open price.
        if "SL" in reason:
            if self.position.direction == 1:
                exit_price = self.position.entry_price - self.position.sl_points
            else:
                exit_price = self.position.entry_price + self.position.sl_points
        else:
            exit_price = current_data['open']
        realized_pnl = ((exit_price - self.position.entry_price) *
                        self.position.direction * self.lot_size)
        realized_pnl -= self.transaction_cost  # Deduct transaction cost.
        trade_return = realized_pnl / self.initial_capital
        total_reward = trade_return
        self.current_capital += realized_pnl
        for trade in reversed(self.trade_log):
            if trade["exit_time"] is None:
                trade.update({
                    "exit_time": current_data.name,
                    "exit_price": exit_price,
                    "pnl": realized_pnl,
                    "reason": reason,
                    "time_in_trade": self.position.time_in_trade
                })
                break
        self.position.reset()
        return total_reward

    def _check_sl_hit(self, current_data: pd.Series) -> bool:
        if self.position.status == "flat":
            return False
        if self.position.direction == 1:  # long
            sl_price = self.position.entry_price - self.position.sl_points
            return current_data['low'] <= sl_price
        else:  # short
            sl_price = self.position.entry_price + self.position.sl_points
            return current_data['high'] >= sl_price

    def _calculate_reward(self, unrealized_pnl: float, current_data: pd.Series) -> float:
        volatility = (current_data['high'] - current_data['low']) / current_data['open']
        reward = (self.unrealized_pnl_weight * (unrealized_pnl / self.initial_capital) -
                  self.time_penalty_weight * (self.position.time_in_trade / 100) -
                  self.volatility_penalty_weight * volatility)
        return float(np.clip(reward, -1.0, 1.0))

    def get_metrics(self) -> dict:
        if not self.trade_log:
            return {}
        winning_trades = [t for t in self.trade_log if t['pnl'] is not None and t['pnl'] > 0]
        losing_trades = [t for t in self.trade_log if t['pnl'] is not None and t['pnl'] <= 0]
        return {
            "instrument": self.instrument_name,
            "initial_capital": self.initial_capital,
            "final_capital": self.current_capital,
            "total_trades": len(self.trade_log),
            "win_rate": len(winning_trades) / len(self.trade_log) if self.trade_log else 0,
            "max_drawdown": self._compute_max_drawdown(),
            "profit_factor": (sum(t['pnl'] for t in winning_trades) /
                              abs(sum(t['pnl'] for t in losing_trades))
                              if losing_trades else np.inf),
        }

    def _compute_max_drawdown(self) -> float:
        equity_curve = [self.initial_capital]
        for trade in self.trade_log:
            if trade['pnl'] is not None:
                equity_curve.append(equity_curve[-1] + trade['pnl'])
        equity_curve = np.array(equity_curve)
        if len(equity_curve) < 2:
            return 0.0
        peak = np.maximum.accumulate(equity_curve)
        dd = (peak - equity_curve) / peak
        return dd.max()

In [49]:
config = {
    # List of instrument configurations. Each instrument must provide:
    # - "name": A unique name for the instrument (e.g., "NIFTY_15M").
    # - "df": A preprocessed pandas DataFrame that includes market columns
    #         (open, high, low, close, StopLoss, Target) as well as the guidance columns:
    #         Signal, candles_to_profit, candles_to_loss.
    # - "lot_size": The size of the trade, e.g., 75 for NIFTY or 1 for Bitcoin.
    # - "transaction_cost": The total cost per trade (brokerage + slippage), e.g., 20.0.
    "instruments": [
        {
            "name": "NIFTY_15M",       # Example: NIFTY index on a 15-minute timeframe.
            "df": df_nifty_15m,         # DataFrame with preprocessed market and guidance data.
            "lot_size": nf_quantity,    # Example: 75.
            "transaction_cost": 20.0    # Example: 20.0 currency units per trade.
        },
        {
            "name": "BANKNIFTY_15M",    # Example: BANKNIFTY index on a 15-minute timeframe.
            "df": df_bnf_15m,
            "lot_size": bnf_quantity,   # Example: 25.
            "transaction_cost": 20.0
        },
        {
            "name": "NIFTY_5M",         # Example: NIFTY index on a 5-minute timeframe.
            "df": df_nifty_5m,
            "lot_size": nf_quantity,    # Example: 75.
            "transaction_cost": 20.0
        },
        {
            "name": "BANKNIFTY_5M",     # Example: BANKNIFTY index on a 5-minute timeframe.
            "df": df_bnf_5m,
            "lot_size": bnf_quantity,   # Example: 25.
            "transaction_cost": 20.0
        },
        {
            "name": "NIFTY_2M",         # Example: NIFTY index on a 2-minute timeframe.
            "df": df_nifty_2m,
            "lot_size": nf_quantity,    # Example: 75.
            "transaction_cost": 20.0
        },
        {
            "name": "BANKNIFTY_2M",     # Example: BANKNIFTY index on a 2-minute timeframe.
            "df": df_bnf_2m,
            "lot_size": bnf_quantity,   # Example: 25.
            "transaction_cost": 20.0
        },
    ],

    # The observation window size.
    # This specifies how many past time steps are included in the history portion of the observation.
    # Example: 30.
    "window_size": 30,

    # Reward Shaping Parameters:
    # These weights adjust the different components of the reward function.

    # Weight for the continuous reward component derived from the normalized unrealized PnL.
    # Example: 0.1 means the unrealized PnL is scaled by 0.1 before being added to the reward.
    "unrealized_pnl_weight": 0.1,

    # Weight for penalizing the time a position is held.
    # This discourages holding positions too long.
    # Example: 0.02 implies a 0.02 penalty per 100 time steps.
    "time_penalty_weight": 0.02,

    # Weight for penalizing market volatility based on the current candle.
    # Volatility is estimated from (high - low) / open.
    # Example: 0.1.
    "volatility_penalty_weight": 0.1,

    # Guidance Reward Parameters (used only during training, not in live trading):
    # These parameters guide the agent based on the future labels that indicate whether a
    # trade at that row would have resulted in a target hit or a stop-loss hit.

    # Bonus factor applied for a correct action when the guidance indicates a target hit.
    # Example: 0.5.
    "target_bonus": 0.5,

    # Penalty factor applied for a correct action if the guidance indicates a stop-loss hit.
    # Example: 0.5.
    "sl_penalty": 0.5,

    # Penalty if the agent’s action does not align with the guidance direction.
    # For instance, if guidance says buy (signal 1 or 2) but the agent sells.
    # Example: 0.5.
    "misalignment_penalty": 0.5,

    # Penalty for missing a trading opportunity.
    # If the agent holds (or issues a close when no position exists) when guidance suggests a trade,
    # a small penalty is applied.
    # Example: 0.05.
    "missed_opportunity_penalty": 0.05
}

In [51]:
# For testing, we use the first instrument's config.
instrument_config = config["instruments"][0]
instrument_config["df"] = df_dummy  # our dummy DataFrame
instrument_config["name"] = "NIFTY_15M"

# Create the environment instance.
env = TradingEnv({**config, **instrument_config})

##########################################
# PPO Agent Implementation
##########################################
# We create a network that outputs both a policy (for discrete actions) and a state value.
def flatten_obs(obs):
    """
    Flattens the observation dictionary by concatenating:
      - market features (shape: [n_features])
      - position vector (shape: [4])
      - flattened history (shape: [window_size * n_features])
    """
    market = obs["market"]
    position = obs["position"]
    history = obs["history"].flatten()
    return np.concatenate([market, position, history])

class PPOPolicy(nn.Module):
    def __init__(self, input_dim, hidden_dim, action_dim):
        super(PPOPolicy, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_policy = nn.Linear(hidden_dim, action_dim)
        self.fc_value = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        policy_logits = self.fc_policy(x)
        value = self.fc_value(x)
        return policy_logits, value

##########################################
# PPO Hyperparameters
##########################################
hidden_dim = 128
action_dim = env.action_space.n
# Determine input dimension from a sample observation.
sample_obs = env.reset()[0]
flat_sample = flatten_obs(sample_obs)
input_dim = flat_sample.shape[0]

policy_net = PPOPolicy(input_dim, hidden_dim, action_dim)
optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)

# PPO training parameters.
ppo_epochs = 4
clip_epsilon = 0.2
gamma = 0.99
lam = 0.95  # for Generalized Advantage Estimation (GAE)
num_updates = 1000
batch_size = 32

##########################################
# PPO Training Loop
##########################################
def compute_gae(rewards, values, masks, gamma=0.99, lam=0.95):
    """
    Computes advantages using Generalized Advantage Estimation (GAE).
    """
    advantages = []
    gae = 0
    values = values + [0]  # Append 0 for the final state.
    for step in reversed(range(len(rewards))):
        delta = rewards[step] + gamma * values[step+1] * masks[step] - values[step]
        gae = delta + gamma * lam * masks[step] * gae
        advantages.insert(0, gae)
    return advantages

all_updates = 0
for update in range(num_updates):
    obs_list = []
    actions_list = []
    logprobs_list = []
    rewards_list = []
    masks_list = []
    values_list = []

    obs = env.reset()[0]
    flat_obs = flatten_obs(obs)
    done = False
    ep_reward = 0
    # Collect trajectory data for one episode.
    while not done:
        obs_tensor = torch.FloatTensor(flat_obs).unsqueeze(0)
        logits, value = policy_net(obs_tensor)
        dist = Categorical(logits=logits)
        action = dist.sample()
        logprob = dist.log_prob(action)

        next_obs, reward, done, _, _ = env.step(action.item())
        next_flat_obs = flatten_obs(next_obs)

        obs_list.append(flat_obs)
        actions_list.append(action.item())
        logprobs_list.append(logprob.item())
        rewards_list.append(reward)
        masks_list.append(0 if done else 1)
        values_list.append(value.item())

        flat_obs = next_flat_obs
        ep_reward += reward

        # Break if trajectory length is large (for safety)
        if len(rewards_list) >= 2048:
            break

    # Compute returns and advantages.
    advantages = compute_gae(rewards_list, values_list, masks_list, gamma, lam)
    returns = [adv + val for adv, val in zip(advantages, values_list)]
    advantages = torch.FloatTensor(advantages)
    returns = torch.FloatTensor(returns)
    obs_tensor = torch.FloatTensor(np.array(obs_list))
    actions_tensor = torch.LongTensor(actions_list)
    old_logprobs_tensor = torch.FloatTensor(logprobs_list)

    # PPO update (multiple epochs over collected data)
    for _ in range(ppo_epochs):
        # For simplicity, we update over the entire trajectory (for larger datasets use minibatching).
        logits, values = policy_net(obs_tensor)
        dist = Categorical(logits=logits)
        new_logprobs = dist.log_prob(actions_tensor)
        entropy = dist.entropy().mean()

        ratio = torch.exp(new_logprobs - old_logprobs_tensor)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()
        value_loss = nn.MSELoss()(values.squeeze(), returns)
        loss = policy_loss + 0.5 * value_loss - 0.01 * entropy

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if update % 10 == 0:
        print(f"Update {update}: Episode Reward = {ep_reward:.3f}")
    all_updates += 1

print("Training completed.")

IndexError: single positional indexer is out-of-bounds